# 01 - Coleta de Dados (OpenAlex) e Construção do Dataset

Este notebook realiza a conexão com a API do OpenAlex, valida o conceito alvo, coleta os artigos conforme os filtros configurados e salva o dataset bruto em `data/raw/`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.coleta import (
    buscar_conceito_openalex,
    coletar_artigos_openalex,
    configurar_pyalex,
    construir_dataframe,
)

## 1. Parâmetros de Configuração
Ajuste os parâmetros abaixo conforme a sua busca.

In [ ]:
# Credenciais opcionais do OpenAlex
EMAIL = ""      # Insira o e-mail cadastrado no OpenAlex
API_KEY = ""    # Insira a chave da API, se possuir

# Parâmetros da consulta
TERMO_CONCEITO = "Industry 4.0"
FALLBACK_CONCEPT_ID = "XXXXXXXXXXX"
YEAR_START = 2011
YEAR_END = 2025

# String de busca modelada como dicionário: {"ÁREA": ["PALAVRA-CHAVE 1", ...]}
KEYWORDS_AREAS = {
    "A": ["B", "C", "D", "E"],
    "X": ["Y", "Z"],
}

# Caminho de saída
OUTPUT_FILE = PROJECT_ROOT / "data" / "raw" / "dataset_bruto.csv"

## 2. Validação do Conceito no OpenAlex

In [ ]:
configurar_pyalex(email=EMAIL, api_key=API_KEY)

top_concept = buscar_conceito_openalex(TERMO_CONCEITO)
if top_concept:
    TARGET_CONCEPT_ID = top_concept["id"]
    print(f"Conceito encontrado: {top_concept['display_name']}")
    print(f"ID Oficial: {TARGET_CONCEPT_ID}")
    print(f"Contagem de obras: {top_concept['works_count']}")
else:
    TARGET_CONCEPT_ID = FALLBACK_CONCEPT_ID
    print(f"Conceito não encontrado via busca; usando fallback: {TARGET_CONCEPT_ID}")

## 3. Coleta de Artigos e Salvamento em CSV

In [ ]:
unique_works, id_to_categories = coletar_artigos_openalex(
    concept_id=TARGET_CONCEPT_ID,
    year_start=YEAR_START,
    year_end=YEAR_END,
    keywords_areas=KEYWORDS_AREAS,
)

df_final = construir_dataframe(unique_works, id_to_categories)
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print(f"Total de artigos processados: {len(df_final)}")
print(f"Arquivo salvo em: {OUTPUT_FILE}")